In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Version (PyTorch):", torch.version.cuda)
print("GPUを認識できているか:", torch.cuda.is_available())
print("利用可能なGPUの数:", torch.cuda.device_count())

PyTorch Version: 2.6.0+cu124
CUDA Version (PyTorch): 12.4
GPUを認識できているか: True
利用可能なGPUの数: 1


In [11]:
print("⏳ [1/3] 15kN Impulse シミュレーションを開始します（数分かかります）...")
!python src/sanshin_analysis_pusg.py
# INPUT_MODE = "impulse" → INPUT_MODE = "sound"にすればsound駆動する

print("✅ [SUCCESS] シミュレーション完了")

⏳ [1/3] 15kN Impulse シミュレーションを開始します（数分かかります）...
Running on Single GPUsanshin_force_sound_real: sim_time=0.3s
--- Simulation start (Single GPU) ---
Grid: 500 x 500 x 500, dt=2.862e-06 s, nt=104840, device=cuda:0
Koma position: real (2L/3)  bridge_y_local=67  (global y=266)
Bridge force scale: 2.000e+03 N/m  accel_per_drive: 7.246e+08 m/s^2/m
Outputs: npy=True, json=True, png=True; cool_sleep_budget=0.0s (every 0 steps x 0.000s)
Output: /app/src
Farfield surface: half=0.32m, stride=2, interval=16, points=155526, frames=6553
Step 1/104840 (  0.0%) - Elapsed:     0.1s, Rem: 12010.8s, End: 15:11:43, Interval:    8.7 step/s, Sleep:    0.0s, MaxVRAM: 4.55 GiB
skipping cudagraphs due to mutated inputs (8 instances). Found from : 
   File "/app/src/sanshin_analysis_pusg.py", line 1260, in fdtd_core_step
    p *= air

Step 10485/104840 ( 10.0%) - Elapsed:   180.1s, Rem:  1620.4s, End: 12:21:32, Interval:   58.3 step/s, Sleep:    0.0s, MaxVRAM: 6.16 GiB
Step 20969/104840 ( 20.0%) - Elapsed:   353

In [14]:
!python ./src/compare_two_npy_polar.py \
    --npy-a ./src/sanshin_force_imp_btr1_real.npy \
    --npy-b ./src/sanshin_force_imp_btr0p8_real.npy \
    --label-a "tension ratio 1.0" \
    --label-b "tension ratio 0.8" \
    --method farfield \
    --frequency-mode mode \
    --out-dir ./polar_comparison_result

frequency mode: mode
polar method: farfield
loading A far-field surface: src/sanshin_force_imp_btr1_real_farfield_surface.json
  surface points used: 155526 (grid stride=1)
loading B far-field surface: src/sanshin_force_imp_btr0p8_real_farfield_surface.json
  surface points used: 155526 (grid stride=1)

Final theoretical modes to plot: 4
  M1: tension ratio 1.0=521.3Hz, tension ratio 0.8=521.3Hz
  M2: tension ratio 1.0=1165.6Hz, tension ratio 0.8=1165.6Hz
  M3: tension ratio 1.0=1879.5Hz, tension ratio 0.8=1879.5Hz
  M4: tension ratio 1.0=2606.4Hz, tension ratio 0.8=2606.4Hz
high-frequency stabilization: freq>=2000Hz, band=90Hz/7 samples, smooth=4
precomputing A far-field spectra...
precomputing B far-field spectra...
Saved: polar_comparison_result/comparison_polar_all_modes_tight_legend.png
Saved: polar_comparison_result/comparison_polar_M1.png
Saved: polar_comparison_result/comparison_polar_M2.png
Saved: polar_comparison_result/comparison_polar_M3.png
Saved: polar_comparison_result/c

In [28]:
!python ./src/compare_spectrum.py

Loading data...
Loaded parameters from sanshin_force_imp_btr1_real_summary.json
Loaded parameters from sanshin_force_sound_btr1_real_summary.json
Modes identified: ['476 Hz', '703 Hz', '1113 Hz', '1723 Hz']
Normalizing by RMS power in range 0-3000.0 Hz...
Baseline removed: -18.57 dB

Saved figure to: fig3_spectrum_analysis.png
Figure(1150x820)


In [12]:
!python ./src/export_microphone_wave.py

Loading data from src/sanshin_force_sound_real_obs_pressure.npy...
Simulation Sampling Rate: 349466.72 Hz (dt: 2.862e-06 s)
Total samples: 104840 (0.300 seconds)
Resampling from 349466.72 Hz to 44100 Hz...
Audio amplitude normalized to max 1.0.
Writing WAV file to src/sanshin_force_sound_real_obs.wav...
Successfully converted to WAV!


In [30]:
!df -h
# 1. ゴミ箱の中にどれくらいデータが溜まっているか確認
!du -sh ~/.local/share/Trash/

# 2. ゴミ箱の中身を完全に消去（一瞬で終わります）
!rm -rf ~/.local/share/Trash/*

# 3. 再びディスク容量を
    # /app フォルダ直下の、どのディレクトリが重いかを調べる（少し時間がかかります）
!du -h --max-depth=1 /app確認（Availが増えているかチェック）
!df -h /app

Filesystem      Size  Used Avail Use% Mounted on
overlay         894G  861G   33G  97% /
tmpfs            64M     0   64M   0% /dev
shm              64M     0   64M   0% /dev/shm
/dev/sda2       894G  861G   33G  97% /app
tmpfs            63G   12K   63G   1% /proc/driver/nvidia
tmpfs            13G  6.9M   13G   1% /run/nvidia-persistenced/socket
tmpfs            63G     0   63G   0% /proc/asound
tmpfs            63G     0   63G   0% /proc/acpi
tmpfs            63G     0   63G   0% /proc/scsi
tmpfs            63G     0   63G   0% /sys/firmware
tmpfs            63G     0   63G   0% /sys/devices/virtual/powercap
du: cannot access '/root/.local/share/Trash/': No such file or directory
du: cannot access '/app確認（Availが増えているかチェック）': No such file or directory
Filesystem      Size  Used Avail Use% Mounted on
/dev/sda2       894G  861G   33G  97% /app


In [29]:
# /app フォルダ直下の、どのディレクトリが重いかを調べる（少し時間がかかります）
!du -h --max-depth=1 /app
# 1. ゴミ箱の中にどれくらいデータが溜まっているか確認
!du -sh ~/.local/share/Trash/

656K	/app/.ipynb_checkpoints
34G	/app/src
1016K	/app/sound_source
32K	/app/.Trash-0
792K	/app/output
11M	/app/polar_comparison_result
34G	/app
du: cannot access '/root/.local/share/Trash/': No such file or directory


In [8]:
# 1. ゴミ箱の中身を完全に消去（51GB分を直接消すので数秒かかります）
!rm -rf /app/.Trash-0/*

# 2. 消去後にディスクの空き容量（Avail）を再確認
!df -h /app

Filesystem      Size  Used Avail Use% Mounted on
/dev/sda2       894G  844G   50G  95% /app
